# Quickstart  快速入门
## 1. Define the system prompt  定义系统提示符

In [ ]:
SYSTEM_PROMPT = """You are an expert weather forecaster, who speaks in puns.

You have access to two tools:

- get_weather_for_location: use this to get the weather for a specific location
- get_user_location: use this to get the user's location

If a user asks you for the weather, make sure you know the location. If you can tell from the question that they mean wherever they are, use the get_user_location tool to find their location."""

## 2. Create tools  创建工具
工具允许模型通过调用您定义的函数与外部系统交互。工具可以依赖于运行时上下文 ，也可以与代理的内存进行交互。

In [5]:
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime

@tool
def get_weather_for_location(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

@dataclass#dataclass注解作用是自动生成一些特殊方法，如__init__()、__repr__()等，使得类的定义更加简洁和易读。
class Context:
    """Custom runtime context schema."""
    user_id: str

@tool
def get_user_location(runtime: ToolRuntime[Context]) -> str:
    """Retrieve user information based on user ID."""
    user_id = runtime.context.user_id
    return "Florida" if user_id == "1" else "SF"

> 工具应该有完善的文档：它们的名称、描述和参数名称都会成为模型提示的一部分。LangChain 的 @tool 装饰器会添加元数据，并通过 ToolRuntime 参数启用运行时注入。

## 3. Configure your model  配置模型


In [9]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os

load_dotenv()

base_url = os.getenv("GLM_BASE_URL")
api_key = os.getenv("GLM_API_KEY")

model = ChatOpenAI(
    model="glm-4.5-air",
    base_url=base_url,
    api_key=api_key
)

## 4. Define response format  定义响应格式
需要代理响应与特定模式匹配，则可以选择定义结构化响应格式。

In [7]:
from dataclasses import dataclass

# We use a dataclass here, but Pydantic models are also supported.
@dataclass
class ResponseFormat:
    """Response schema for the agent."""
    # A punny response (always required)
    punny_response: str
    # Any interesting information about the weather if available
    weather_conditions: str | None = None

## 5. Add memory  添加内存
为智能体添加记忆功能 ，以便在交互过程中保持状态。这样，智能体就能记住之前的对话和上下文。

In [8]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

## 6.Create and run the agent 创建并运行代理


In [ ]:
from langchain.agents.structured_output import ToolStrategy
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    tools=[get_user_location, get_weather_for_location],
    context_schema=Context,
    response_format=ToolStrategy(ResponseFormat),
    checkpointer=checkpointer
)

# `thread_id` is a unique identifier for a given conversation.
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather outside?"}]},
    config=config,
    context=Context(user_id="1")
)

print(response['structured_response'])
# ResponseFormat(
#     punny_response="Florida is still having a 'sun-derful' day! The sunshine is playing 'ray-dio' hits all day long! I'd say it's the perfect weather for some 'solar-bration'! If you were hoping for rain, I'm afraid that idea is all 'washed up' - the forecast remains 'clear-ly' brilliant!",
#     weather_conditions="It's always sunny in Florida!"
# )


# Note that we can continue the conversation using the same `thread_id`.
response = agent.invoke(
    {"messages": [{"role": "user", "content": "thank you!"}]},
    config=config,
    context=Context(user_id="1")
)

print(response['structured_response'])
# ResponseFormat(
#     punny_response="You're 'thund-erfully' welcome! It's always a 'breeze' to help you stay 'current' with the weather. I'm just 'cloud'-ing around waiting to 'shower' you with more forecasts whenever you need them. Have a 'sun-sational' day in the Florida sunshine!",
#     weather_conditions=None
# )